In [1]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import argparse
from tqdm import tqdm
import matplotlib.pyplot as plt 
import seaborn as sns

from lib.networks_prototype import DINOv3_Prototype
from utils.dataloader import test_dataset

# ==========================================
# 1. 高對比度可視化工具 (High Contrast Vis)
# ==========================================
def save_visualization(original_image, true_mask, binary_pred, k_class_pred, output_path, fg_num, bg_num):
    """
    生成四聯圖：原圖 | 真值 | 二元預測 | K類子預測
    original_image: (H, W, 3) uint8 BGR
    """
    h, w, _ = original_image.shape
    
    def _resize(m):
        if m.shape != (h, w):
            return cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        return m
    
    true_mask = _resize(true_mask)
    binary_pred = _resize(binary_pred)
    k_class_pred = _resize(k_class_pred)

    font = cv2.FONT_HERSHEY_SIMPLEX
    def add_title_and_border(img, txt):
        img = cv2.copyMakeBorder(img, 40, 0, 0, 0, cv2.BORDER_CONSTANT, value=[255, 255, 255])
        cv2.putText(img, txt, (10, 30), font, 0.8, (0, 0, 0), 2, cv2.LINE_AA)
        return img

    # [A] Original
    vis_original = add_title_and_border(original_image.copy(), "Original")

    # [B] Ground Truth (Yellow = FG)
    vis_gt = original_image.copy()
    overlay = vis_gt.copy()
    overlay[true_mask == 1] = [0, 255, 255] # Yellow
    vis_gt = cv2.addWeighted(overlay, 0.5, vis_gt, 0.5, 0)
    vis_gt = add_title_and_border(vis_gt, "Ground Truth")

    # [C] Binary Prediction (Green = Pred)
    vis_binary = original_image.copy()
    overlay = vis_binary.copy()
    overlay[binary_pred == 1] = [0, 255, 0] # Green
    vis_binary = cv2.addWeighted(overlay, 0.5, vis_binary, 0.5, 0)
    vis_binary = add_title_and_border(vis_binary, "Binary Pred")
    
    fg_palette = np.array([
        [0, 0, 255], [0, 165, 255], [0, 255, 255], [255, 0, 255],
        [0, 255, 128], [255, 0, 128], [128, 0, 255], [0, 64, 255],
    ], dtype=np.uint8)

    bg_palette = np.array([
        [255, 0, 0], [255, 255, 0], [128, 30, 170], [0, 130, 100],
        [128, 128, 128], [64, 64, 64], [128, 64, 0], [0, 128, 128],
    ], dtype=np.uint8)

    k_colored_map = np.zeros((h, w, 3), dtype=np.uint8)    
    for fg_id in range(min(len(fg_palette), fg_num)):
        k_colored_map[k_class_pred == fg_id] = fg_palette[fg_id]
    for bg_id in range(min(len(bg_palette), bg_num)):
        k_colored_map[k_class_pred == (fg_num + bg_id)] = bg_palette[bg_id]
    vis_k = add_title_and_border(k_colored_map, "K-Class Prototypes")

    combined = np.concatenate((vis_original, vis_gt, vis_binary, vis_k), axis=1)
    cv2.imwrite(output_path, combined)

def tensor_to_bgr_image(tensor):
    """將 Normalize 過的 Tensor 還原回 BGR 圖片"""
    tensor = tensor.squeeze(0).cpu()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    img = tensor * std + mean
    img = torch.clamp(img, 0, 1)
    img_np = img.permute(1, 2, 0).numpy()
    
    img_bgr = (img_np * 255).astype(np.uint8)
    img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_RGB2BGR)
    return img_bgr

# ==========================================
# 2. 原型熱力圖分析 (Prototype Analysis)
# ==========================================
def analyze_prototypes(model, output_dir, num_fg, num_bg):
    print("\n正在分析 DINOv3 原型相似度...")
    
    if hasattr(model, 'prototypes'):
        prototypes = model.prototypes.data.cpu().numpy()
    else:
        print("警告: 模型沒有 'prototypes' 屬性，跳過分析。")
        return

    norms = np.linalg.norm(prototypes, axis=1, keepdims=True)
    prototypes_norm = prototypes / (norms + 1e-8)
    similarity_matrix = np.dot(prototypes_norm, prototypes_norm.T)
    
    plt.figure(figsize=(10, 8))
    labels = [f'FG_{i+1}' for i in range(num_fg)] + [f'BG_{i+1}' for i in range(num_bg)]
    
    if len(labels) != similarity_matrix.shape[0]:
        labels = [str(i) for i in range(similarity_matrix.shape[0])]

    sns.heatmap(similarity_matrix, 
                annot=True, fmt=".2f", 
                xticklabels=labels, yticklabels=labels,
                cmap="coolwarm", vmin=-1, vmax=1)
    
    plt.title("DINOv3 Learned Prototypes Cosine Similarity")
    plt.tight_layout()
    
    save_path = os.path.join(output_dir, "prototypes_similarity_heatmap.png")
    plt.savefig(save_path)
    plt.close()
    print(f"原型相似度熱力圖已儲存至: {save_path}")

# ==========================================
# 3. 核心評估函式 (Evaluation Loop)
# ==========================================
def test_and_visualize(model, data_path, cfg):
    image_root = '{}/images/'.format(data_path)
    gt_root = '{}/masks/'.format(data_path)
    
    test_loader = test_dataset(image_root, gt_root, cfg['image_size'])
    
    model.eval()
    num1 = test_loader.size
    DSC = 0.0
    
    output_dir = cfg['output_dir']
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"[INFO] Testing on {num1} images from {data_path}...")
    
    with torch.no_grad():
        for i in tqdm(range(num1)):
            image, gt, name = test_loader.load_data()
            
            gt = np.asarray(gt, np.float32)
            gt /= (gt.max() + 1e-8)
            image = image.cuda()

            # DINOv3_Prototype 回傳: (logits_high, similarity_map_high)
            logits_binary, logits_k_class = model(image)
            
            # (FG - BG) * 10
            res = (logits_binary[:, 1:2, :, :] - logits_binary[:, 0:1, :, :]) * 10.0
            res = F.interpolate(res, size=gt.shape, mode='bilinear', align_corners=False)
            res = res.sigmoid().data.cpu().numpy().squeeze()
            res = (res - res.min()) / (res.max() - res.min() + 1e-8)
            
            input_flat = np.reshape(res, (-1))
            target_flat = np.reshape(gt, (-1))
            intersection = (input_flat * target_flat)
            smooth = 1
            dice = (2 * intersection.sum() + smooth) / (res.sum() + gt.sum() + smooth)
            DSC = DSC + dice

            # 進行視覺化
            vis_id = [1, 2, 10, 12, 50, 51, 100, 101, 150, 151, 200, 
                      201, 300, 301, 400, 401, 500, 501, 550, 551, 600, 601, 650, 651, 700, 701]
            if i in vis_id:
                vis_img_bgr = tensor_to_bgr_image(image)
                pred_binary_mask = (res > 0.5).astype(int)
            
                sim_map_resized = F.interpolate(logits_k_class, size=gt.shape, mode='bilinear', align_corners=False)
                pred_k_mask = sim_map_resized.argmax(dim=1).squeeze(0).cpu().numpy()
                gt_binary_mask = (gt > 0.5).astype(int)

                save_path = os.path.join(output_dir, f"{name}")
                save_visualization(
                    original_image=vis_img_bgr,
                    true_mask=gt_binary_mask,
                    binary_pred=pred_binary_mask,
                    k_class_pred=pred_k_mask,
                    output_path=save_path,
                    fg_num=cfg['model_cfg']['num_fg_prototypes'],
                    bg_num=cfg['model_cfg']['num_bg_prototypes']
                )

    avg_dice = DSC / num1
    return avg_dice

# ==========================================
# 4. Main Entry
# ==========================================
def main():
    config = {
        'data_path': '/home/U116med/data/polyp/TestDataset/test', # 請修改為您要測試的資料集路徑
        'model_path': '/home/U116med/wch_code/dino_v3/models/prototype_cascade/vits16plus/baseline_prototype_fb_44_fine/best.pth', # 請修改為您的權重路徑
        'image_size': 352,
        'output_dir': './dinov3_visualizations/vits16plus/fb44_fine/',
        'model_cfg': {
            'num_fg_prototypes': 4, # 需與訓練時設定一致
            'num_bg_prototypes': 4,
            'backbone_type': 'vits16plus'
        }
    }

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # 1. 載入 DINOv3 原型模型
    model = DINOv3_Prototype(
        num_fg=config['model_cfg']['num_fg_prototypes'], 
        num_bg=config['model_cfg']['num_bg_prototypes'],
        backbone_type=config['model_cfg']['backbone_type']
    ).to(device)

    # 2. 載入權重
    if os.path.exists(config['model_path']):
        print(f"Loading weights: {config['model_path']}")
        checkpoint = torch.load(config['model_path'], map_location=device, weights_only=True)
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'], strict=True)
        else:
            model.load_state_dict(checkpoint, strict=True)
    else:
        print(f"Error: Model path {config['model_path']} does not exist.")
        return

    # 3. 執行測試與可視化
    avg_dice = test_and_visualize(model, config['data_path'], config)
    
    print("\n" + "="*40)
    print(f"Dataset processed.")
    print(f"Average Dice Score: {avg_dice:.4f}")
    print("="*40)

    analyze_prototypes(model, config['output_dir'], 
                       config['model_cfg']['num_fg_prototypes'], 
                       config['model_cfg']['num_bg_prototypes'])

if __name__ == '__main__':
    main()

Device: cuda
Loading DINOv3: dinov3_vits16plus


Using cache found in /home/U116med/.cache/torch/hub/facebookresearch_dinov3_main


Loading weights: /home/U116med/wch_code/dino_v3/models/prototype_cascade/vits16plus/baseline_prototype_fb_44_fine/best.pth
[INFO] Testing on 798 images from /home/U116med/data/polyp/TestDataset/test...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 798/798 [00:19<00:00, 41.11it/s]



Dataset processed.
Average Dice Score: 0.8174

正在分析 DINOv3 原型相似度...
原型相似度熱力圖已儲存至: ./dinov3_visualizations/vits16plus/fb44_fine/prototypes_similarity_heatmap.png
